# Criação da Tabela Fato — Assistência Estudantil (UFPB, Campus I)

Este notebook constrói a **tabela fato** de assistência estudantil a partir
dos dados do SEDAP+ (Censo da Educação Superior 2024), já usando as chaves
das dimensões geradas em `1_criacao_dimensoes.ipynb` / `src/build_dimensions.py`.

**Colunas finais da fato:**
`ID_CURSO`, `ID_SEXO`, `ID_RACA`, `ID_TURNO`, `TP_LEI_COTAS`,
`IN_ACAO_AFIRMATIVA`, `IN_APOIO_SOCIAL`, `IN_APOIO_ALIMENTACAO`,
`IN_APOIO_MORADIA`, `IN_APOIO_TRANSPORTE`, `IN_APOIO_MATERIAL_DIDATICO`,
`IN_APOIO_BOLSA_PERMANENCIA`, `IN_APOIO_BOLSA_TRABALHO`, `TOTAL_ALUNOS`,
`RECEBE_AUXILIO`, `IDPNA`, `TOTAL_IDPNA`

**Etapas do notebook:**
1. Importação das bibliotecas
2. Leitura da tabela fato
3. Filtrar Campus I
4. Renomear colunas
5. Atualizar a dimensão turno (aplicar na fato)
6. Tratar valores nulos
7. Criar `RECEBE_AUXILIO`
8. Validar `IN_ACAO_AFIRMATIVA`
9. Criar `IDPNA`
10. Criar `TOTAL_IDPNA`
11. Validações
12. Exportação do CSV final

## 1. Importação das bibliotecas

In [ ]:
import pandas as pd

from src.facts.fato_assistencia import criar_fato_assistencia

In [ ]:
fato = pd.read_csv("../data/raw/FATO_ACAO.csv")

In [ ]:
fato = criar_fato_assistencia(fato)

In [ ]:
print("Número de registros:", len(fato))
print("Cursos:", fato["ID_CURSO"].nunique())

display(fato.head())

In [ ]:
fato.isna().sum()

In [ ]:
fato["RECEBE_AUXILIO"].value_counts()

In [ ]:
fato["IDPNA"].value_counts()

In [ ]:
print(fato["ID_CURSO"].nunique())
print(fato["ID_SEXO"].unique())
print(fato["ID_RACA"].unique())
print(fato["ID_TURNO"].unique())

In [ ]:
fato["RECEBE_AUXILIO"] = fato["IN_APOIO_SOCIAL"].astype(int)

In [ ]:
print(fato["IN_ACAO_AFIRMATIVA"].value_counts(dropna=False))

# Caso existam nulos, tratamos como "não ingressou por ação afirmativa" (0)
fato["IN_ACAO_AFIRMATIVA"] = (
    fato["IN_ACAO_AFIRMATIVA"]
    .fillna(0)
    .astype(int)
)

In [ ]:
fato["IDPNA"] = (
    (fato["IN_ACAO_AFIRMATIVA"] == 1) &
    (fato["RECEBE_AUXILIO"] == 0)
).astype(int)

In [ ]:
fato["TOTAL_IDPNA"] = (
    fato["IDPNA"] *
    fato["TOTAL_ALUNOS"]
)

In [ ]:
display(fato.head(10))

In [ ]:
print(fato.columns)

In [ ]:
fato.isnull().sum()

In [ ]:
print("Cursos:", fato["ID_CURSO"].nunique())

In [ ]:
print(fato["IDPNA"].value_counts())

In [ ]:
print(fato["RECEBE_AUXILIO"].value_counts())

In [ ]:
print("Total de alunos:", fato["TOTAL_ALUNOS"].sum())

In [ ]:
fato.groupby("ID_CURSO")[["TOTAL_ALUNOS", "TOTAL_IDPNA"]].sum().head(10)

In [ ]:
# Confere se há algum curso presente na dim_curso (Campus I) que não aparece na fato
dim_curso_check = pd.read_csv("../data/processed/dim_curso_completo.csv")

cursos_dim = set(dim_curso_check["ID_CURSO"])
cursos_fato = set(fato["ID_CURSO"])

faltando = cursos_dim - cursos_fato

print("Quantidade:", len(faltando))
print(sorted(faltando))

In [ ]:
dim_curso_check[
    dim_curso_check["ID_CURSO"].isin(faltando)
][["ID_CURSO", "CURSO"]]

In [ ]:
fato.to_csv(
    "../data/processed/fato_final.csv",
    index=False,
    encoding="utf-8-sig"
)